In [0]:
from pyspark.sql.functions import explode, col, lit, when
from delta.tables import DeltaTable
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

dbutils.widgets.text("team_id", "") 
team_id = dbutils.widgets.get("team_id")
logger.info("id from adf: " + team_id)

df = spark.read.json("abfss://faceit@storageonline23.dfs.core.windows.net/teams")

f1 = df.select(
    col("avatar"),
    col("faceit_url"),
    col("name"),
    col("nickname"),
    col("team_id"),
    col("team_type"),
    explode("members").alias("member")
)

f1 = f1.select(
    "avatar",
    "faceit_url",
    "name",
    "nickname",
    "team_id",
    "team_type",
    col("member.user_id").alias("user_id"),
    col("member.nickname").alias("player_nickname"),
    col("member.country").alias("country")
)

delta_table = DeltaTable.forName(spark, "ddca26online23.default.silver_team_history")
logger.info(f"Target rows before merge: {delta_table.toDF().count()}")

delta_table.alias("tgt").merge(
    f1.alias("src"),
    "tgt.team_id = src.team_id AND tgt.user_id = src.user_id"
).whenNotMatchedInsertAll().execute()

logger.info(f"Target rows after merge: {delta_table.toDF().count()}")

